# Steps for the exam
#### 1. Analyze the data and design appropriate transformations/preprocessing.  

#### 2. Train a model (training must happen on-the-fly when main.py is executed; it is possible, anyway, to run a grid search for parameter tuning and then use in the main.py file the desired model configuration).

#### 3. Predict labels for the provided test set.

#### 4. Save predictions to submission.csv (the file to be graded).

---

# Common steps for each model

#### 1. data inspection  
- import dataset as dataframe
- display the whole dataframe to have an idea and to have a generic look
- inspect shape (n rows and n column)
- inspect data type of each column --> see if you have labels  
###### (in general in the traccia of the exam it is written what kind of problem it is so you know if you have labels or not, usually you do)  

- check if columns are dirty or not --> like if there's dirty text --> clean by stripping / lower / replace --> so you spot hidden empty / fake not nan rows --> **replace with Nan**
- check if columns have nan-like values / values equivalent to (es. 0000-00-00 in timestamp) --> **replace with Nan**
- drop clear useless columns: almost-constant col + ID columns
- now that the data should be clean and actual data --> **MANAGE NAN** --> this changes for each model, figure it out:
    - analyse columns:
        - inspect the % of Nan in each column
        - understand how important is that column
            - if it's crucial --> KNNImpute if numerical / simplimpute if numerical or categorical / replace nan with '' if plaintext and needs TF-IDF
            - if it's less crucial think if to drop it / impute / replace with mean or 0
    
    - analyse rows:
        - say that a row is filled with Nans --> you drop it of course because it's garbage for the training
        - say that a crucial column has like 20 nan values --> you drop those rows
        - **BUT**: THE ROW FILTERING/DROPPING SHOULD BE DONE **ONLY** IN TRAINING, IN THE EVALUATION WHEN YOU CLEAN YOU CAN'T DROP ROWS OTHERWISE THE PREDICTED LABELS / TARGET VALUES WON'T MATCH THE SOLUTIONS --> KEEP ALL ROWS IN THE EVALUATION.
        - while if you drop whole columns in the training --> you can do for the evaluation too

- now data is cleaned and we handled Nan values accordingly
- plot data frequency + display column count values --> spot almost constant columns + columns with just one value + columns with outsider / anomalies + spot nan-like values (es. 0000-00-00 timestamp)

- based on the data type of each column --> think of the preprocess pipeline each (useful) column (if necessary):
    - SimpleImputer --> numbers and categories, but dumber / more naive
    - KNNImputer --> only numbers
    - OneHotEncode / TargetEncode --> categorical data + target encoder especially if you have many unique values
    - OrdinalEncode --> ordinal columns + timestamps
    - Tokenizer + lemmatize + TF-IDF + CSV --> plaintext cols
    - StandardScaler --> numeric data that needs to be scaled
    - MinMaxScaler --> 
    - PolynomialFeatures --> ONLY COLUMNS WITH HIGHEST CORRELATION WITH THE TARGET, otherwise the dimension explodes

- feed the preprocessed data to the model
- train + tune it
- predict
- evaluate

- in the exam you have an evaluation set, but without labels --> keep a small piece (10/20 %) of the train data an use it as evaluation, don't train on that piece --> useful to see the model actual score.
- At the end don't forget to add that data you left as evaluation to the training though.

---

# Regression
- after preprocessing data depending on the data type
- build the model:
    - **LinearRegression**
    - **Ridge**
    - **Lasso**

- tune the hyperparameters:
    - GridSearchCV
    - RidgeCV
    - LassoCV

- find the best model (automatically trained by the gridsearch)
- import, clean (WITHOUT DROPPING ROWS), preprocess (WITHOUT RE-BUILDING THE TOOLS BUT RE-USING THE ONES CREATED IN THE TRAINING) evaluation set
- use the already trained best model to predict the cleaned + preprocessed evaluation set:
    - regression evaluation:
        - Mean Squared Error **MSE**, Root Mean Squared Error **RMSE**, Mean Absolute Error **MAE**
        - **R SQUARED $R^2$** 
- move the predictions to a csv

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, root_mean_squared_log_error, mean_absolute_error, r2_score, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
file_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/train_dataset.csv'

df = pd.read_csv(file_path)
display(df)


- data inspection

In [ ]:
df.info()
df.shape

In [ ]:
df.describe()

- Numerical data hasn't mean 0 and std 1: NEEDS TO BE SCALED

In [ ]:
df.columns

- regression problem, target is numerical
- mix of data types: numbers and not --> numeric, ordinal, categorical
- explicit nan values
- shape (6000, 59)
- no immediatly useless columns

- look for hidden Nan values

In [ ]:
for col in df:
    print(df[col].value_counts())

- numeric columns seems ok, no Nan-like values
- ordinal columns seems ok, no Nan-like values and not dirty text
- categorical columns seems ok, no Nan-like values and not dirty text

No need to clean, inspect only explicit Nan values

In [ ]:
na_count_col = df.isna().sum(axis=0)
na_per_col = (df.isna().mean(axis=0))*100

for col in df:
    print(f'{col} : {na_count_col[col]} --> {na_per_col[col]}%')



- many many Nans

- since my preprocessing idea is:
    - numerical columns --> scale them + feed them to a regressor --> replace with 0 --> **NO**, ZERO IS A NUMBER AND AFTER YOU SCALE THESE NUMBERS SO EVEN THESE 0 BECOME A WEIRD VALUE AND THE MODEL LEARNS FROM THEM --> **impute median + add a missing value column**
    - categorical columns --> OneHotEncode --> replace with 'unknown' (no need for additional missing value column)
    - ordinal column --> OrdinalEncode --> the encoder deals with Nan and can replace them with -1 (no need for additional missing value column)

- inspect Nan per row

In [ ]:
n_na_row = df.isna().sum(axis=1)
# print(n_na_row)

count = 0
for row_idx, na in enumerate(n_na_row):
    if na:
        print(f'{row_idx}, {na}, {(na/df.shape[1])}%')
        count += 1

print(df.isna().sum(axis=1).sum())

In [9]:
# divide columns based on types
num_col = df.loc[:, 'cont_0':'cont_29']

ord_col = df.loc[:, 'ord_0':'ord_19']

cat_col = df.loc[:, 'cat_0':'cat_7']


In [10]:
# manage nans, later add it to the preprocessing pipeline so it's cleaner

# numeric columns --> build missing value column + impute Nan with median
for col in num_col:
    num_col[f"missing_{col}"] = (num_col[col].isna()).astype(int)

num_imputer = SimpleImputer(missing_values=np.nan, strategy='median')
num_col_prep = num_imputer.fit_transform(num_col)
num_col_prep = pd.DataFrame(num_col_prep, columns=num_col.columns)


# ordinal columns --> impute with constant value -1
# ord_col.fillna(-1, inplace=True)
ord_imputer = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1)
ord_col_prep = ord_imputer.fit_transform(ord_col)
ord_col_prep = pd.DataFrame(ord_col_prep, columns=ord_col.columns)

# categorical columns --> impute with constant value 'unknown'
# cat_col.fillna('uknown', inplace=True)
cat_imputer = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown')
cat_col_prep = cat_imputer.fit_transform(cat_col)
cat_col_prep = pd.DataFrame(cat_col_prep, columns=cat_col.columns)



- plot distribution numerical columns

In [ ]:
# plot distribution to identify eventual useless col

plt.figure(figsize=(12,20))
for plt_idx, num_col in enumerate(num_col_prep.loc[:, 'cont_0':'cont_29']):
    
    plt.subplot(10,3, plt_idx+1)
    sns.histplot(x=num_col_prep[num_col], bins='auto', kde=True)
    plt.xlabel('values')
    plt.ylabel('frequency')
    plt.title(f'histplot {num_col}')

plt.tight_layout()
plt.show()


In [ ]:
for col in num_col_prep:
    print(num_col_prep[col].value_counts())

- those weird spikes are normal, those are the previous Nan values that have all been replaced with the median
- but they could actually be harmful that's why we added the additional missing value column --> **OR IF THE SCORING IS BAD USE KNN IMPUTER (no additional column)**

In [ ]:
# plot ordinal values
plt.figure(figsize=(20,25))
plot_ord_df = ord_col_prep.replace(-1, 'unknown')

for plt_idx, col in enumerate(plot_ord_df):
    col_values = plot_ord_df[col].values

    plt.subplot(10,2, plt_idx+1)
    sns.histplot(x=col_values, bins=len(col_values))
    # plt.hist(x=ord_col_prep[col].values, bins=len(ord_col_prep[col].values))
    
    plt.xlabel('values')
    plt.ylabel('frequency')
    plt.title(f'histplot {col}')

plt.tight_layout()
plt.show()
    

- the stinkiest columns are ord_0, ord_3, ord_5, ord_10, ord_17, ord_18 --> the highest frequency values is 'unknown', which is -1

In [ ]:
# plot categorical values

plt.figure(figsize=(13,18))
for plt_idx, col in enumerate(cat_col_prep):
    col_values = cat_col_prep[col].values

    plt.subplot(4,2, plt_idx+1)
    sns.histplot(x=col_values, bins=len(col_values))
    plt.xlabel('values')
    plt.ylabel('frequency')
    plt.title(f'histplot {col}')
    plt.xticks(rotation=30)

plt.tight_layout()
plt.show()


- cat_0, cat_3, cat_7 are very stinky

---

- build cleaned dataframe after managing Nans

In [15]:
# rebuild dataframe
df_cleaned = pd.concat(objs=[num_col_prep, ord_col_prep, cat_col_prep], axis=1)     # axis = 1 to merge horizontally

# recheck Nan values per col, there shouldn't be any
na_count_col = df_cleaned.isna().sum(axis=0)
na_per_col = (df_cleaned.isna().mean(axis=0))*100

for col in df_cleaned:
    if na_count_col[col]:
        print(f'{col} : {na_count_col[col]} --> {na_per_col[col]}%')


na_count_row = df_cleaned.isna().sum(axis=1)
value = na_count_row.values
row_idx = na_count_row.index

for v,i in zip(value,row_idx):
    if v:
        print(i,v)

# no more nan values :)

---

- now that we imputed --> more preprocessing
- the idea is:
    - numerical columns --> scale them
    - ordinal columns --> ordinal encode them
    - categorical columns --> one hot encode

# workflow
- split data into train + evaluation
- Build full preprocessing pipeline:
    
    - numerical columns pipeline:
        - simplimputer median (could consider a KNNimputer) + build missing numerical data columns **USING THE IMPUTER**
        - standard scaler
    
    - ordinal columns pipeline:
        - simpleimputer -1
        - ordinalencoder
    
    - categorical columns pipeline:
        - simpleimputer 'unknow'
        - OneHotEncoding
    
    - tie all these 3 preprocessing pipelines together with a columntransformer
    - build a final full pipeline:
        - preprocesing pipeline (3 pipelines tied together)
        - regression model:
            - simpleregressor
            - Ridge
            - Lasso
    - add a grisearch with cross validation --> LassoCV or RidgeCV

    - get best model
    - predict on x_ev
    - evaluate

In [ ]:
df

In [ ]:
# THE SIMPLEIMPUTER AUTOMATICALLY DOES THIS !!!!
# num_col = df.loc[:, 'cont_0':'cont_29']
# for col in num_col:
#     df[f"missing_{col}"] = (num_col[col].isna()).astype(int)

targets = df['target']
data = df.drop(columns='target')

x_train, x_ev, y_train, y_ev = train_test_split(data, targets, test_size=0.2, train_size=0.8, random_state=42, shuffle=True)

num_col = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']

ord_col = ['ord_0', 'ord_1','ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6',
           'ord_7', 'ord_8', 'ord_9', 'ord_10', 'ord_11', 'ord_12',
           'ord_13', 'ord_14', 'ord_15', 'ord_16',
           'ord_17', 'ord_18', 'ord_19']

cat_col = ['cat_0', 'cat_1', 'cat_2', 'cat_3',
       'cat_4', 'cat_5', 'cat_6', 'cat_7']

num_pipeline = Pipeline(steps=[
    (
        'num_imputer',
        SimpleImputer(missing_values=np.nan, strategy='median', add_indicator=True)     # add_indicator=True builds the missing values column
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipeline = Pipeline(steps=[
    # (
    #     'ord_imputer',
    #     SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1)        
    # )
    (
        'OE',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value = -1)  # be careful about the fill_value as the whole columns are stings and not numbers ...
    )
])

cat_pipeline = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OHE',
        OneHotEncoder(handle_unknown='ignore')
    )
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('numeric', num_pipeline, num_col)
    ,
    ('ordinal', ord_pipeline, ord_col)
    ,
    ('categorical', cat_pipeline, cat_col)
    ])


full_pipeline = Pipeline(steps=[
    ('preprocessing',preprocessing_pipeline),
    ('regressor', LinearRegression(n_jobs=-1))
])

full_pipeline.fit(x_train, y_train)
y_pred = full_pipeline.predict(x_ev)

print(r2_score(y_ev, y_pred))


That's ... pretty bad :)

- Let's try using a KNNimputer for numerical cols
- Use a ridge + a gridsearch to find n_neighbours and ridge hyperparameters

In [ ]:
targets = df['target']
data = df.drop(columns='target')

x_train, x_ev, y_train, y_ev = train_test_split(data, targets, test_size=0.2, train_size=0.8, random_state=42, shuffle=True)

num_col = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']

ord_col = ['ord_0', 'ord_1','ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6',
           'ord_7', 'ord_8', 'ord_9', 'ord_10', 'ord_11', 'ord_12',
           'ord_13', 'ord_14', 'ord_15', 'ord_16',
           'ord_17', 'ord_18', 'ord_19']

cat_col = ['cat_0', 'cat_1', 'cat_2', 'cat_3',
       'cat_4', 'cat_5', 'cat_6', 'cat_7']

num_pipeline = Pipeline(steps=[
    (
        'num_imputer',
        KNNImputer(missing_values=np.nan)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipeline = Pipeline(steps=[
    # (
    #     'ord_imputer',
    #     SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1)        
    # )
    (
        'OE',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value = -1)  # be careful about the fill_value as the whole columns are stings and not numbers ...
    )
])

cat_pipeline = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OHE',
        OneHotEncoder(handle_unknown='ignore')
    )
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('numeric', num_pipeline, num_col)
    ,
    ('ordinal', ord_pipeline, ord_col)
    ,
    ('categorical', cat_pipeline, cat_col)
    ])


full_pipeline = Pipeline(steps=[
    ('preprocessing',preprocessing_pipeline),
    ('regressor', Ridge(random_state=42))
])



param_grid = {
    'regressor__alpha': [5,15,50],
    'preprocessing__numeric__num_imputer__n_neighbors' : [3,5,10]
}

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='r2')
grid.fit(x_train, y_train)
print(grid.best_params_)
best_model = grid.best_estimator_

y_pred = best_model.predict(x_ev)
print(r2_score(y_ev, y_pred))

- ok try to fix n_neighbors = 3
- perform a:
    - RidgeCV
    - LassoCV

In [ ]:
targets = df['target']
data = df.drop(columns='target')

x_train, x_ev, y_train, y_ev = train_test_split(data, targets, test_size=0.2, train_size=0.8, random_state=42, shuffle=True)

num_col = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']

ord_col = ['ord_0', 'ord_1','ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6',
           'ord_7', 'ord_8', 'ord_9', 'ord_10', 'ord_11', 'ord_12',
           'ord_13', 'ord_14', 'ord_15', 'ord_16',
           'ord_17', 'ord_18', 'ord_19']

cat_col = ['cat_0', 'cat_1', 'cat_2', 'cat_3',
       'cat_4', 'cat_5', 'cat_6', 'cat_7']

num_pipeline = Pipeline(steps=[
    (
        'num_imputer',
        KNNImputer(missing_values=np.nan, n_neighbors=3)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipeline = Pipeline(steps=[
    # (
    #     'ord_imputer',
    #     SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1)        
    # )
    (
        'OE',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value = -1)  # be careful about the fill_value as the whole columns are stings and not numbers ...
    )
])

cat_pipeline = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OHE',
        OneHotEncoder(handle_unknown='ignore')
    )
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('numeric', num_pipeline, num_col)
    ,
    ('ordinal', ord_pipeline, ord_col)
    ,
    ('categorical', cat_pipeline, cat_col)
    ])


full_pipeline = Pipeline(steps=[
    ('preprocessing',preprocessing_pipeline),
    ('regressor', RidgeCV(alphas=[0.1, 1, 5, 10, 15, 20], cv=4))
])


full_pipeline.fit(x_train, y_train)
y_pred = best_model.predict(x_ev)
print(r2_score(y_ev, y_pred))
print(mean_absolute_error(y_ev, y_pred))
print(mean_absolute_error(y_ev, y_pred))
print(full_pipeline.named_steps['regressor'].alpha_)

---

# Model 2
- same but uses Lasso

In [ ]:
targets = df['target']
data = df.drop(columns='target')

x_train, x_ev, y_train, y_ev = train_test_split(data, targets, test_size=0.2, train_size=0.8, random_state=42, shuffle=True)

num_col = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']

ord_col = ['ord_0', 'ord_1','ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6',
           'ord_7', 'ord_8', 'ord_9', 'ord_10', 'ord_11', 'ord_12',
           'ord_13', 'ord_14', 'ord_15', 'ord_16',
           'ord_17', 'ord_18', 'ord_19']

cat_col = ['cat_0', 'cat_1', 'cat_2', 'cat_3',
       'cat_4', 'cat_5', 'cat_6', 'cat_7']

num_pipeline = Pipeline(steps=[
    (
        'num_imputer',
        KNNImputer(missing_values=np.nan, n_neighbors=3)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipeline = Pipeline(steps=[
    # (
    #     'ord_imputer',
    #     SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1)        
    # )
    (
        'OE',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value = -1)  # be careful about the fill_value as the whole columns are stings and not numbers ...
    )
])

cat_pipeline = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OHE',
        OneHotEncoder(handle_unknown='ignore')
    )
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('numeric', num_pipeline, num_col)
    ,
    ('ordinal', ord_pipeline, ord_col)
    ,
    ('categorical', cat_pipeline, cat_col)
    ])


full_pipeline = Pipeline(steps=[
    ('preprocessing',preprocessing_pipeline),
    ('regressor', LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10], cv=4, max_iter=10000))
])


full_pipeline.fit(x_train, y_train)
y_pred = best_model.predict(x_ev)
print(r2_score(y_ev, y_pred))
print(mean_absolute_error(y_ev, y_pred))
print(mean_absolute_error(y_ev, y_pred))
print(full_pipeline.named_steps['regressor'].alpha_)

---

- fuck it try to predict the test data
    - REMEMBER TO TRAIN ON THE WHOLE DATASET, DON'T USE EVALUATION ANYMORE!!

In [ ]:
targets = df['target']
data = df.drop(columns='target')

x_train = data
y_train = targets

num_col = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']

ord_col = ['ord_0', 'ord_1','ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6',
           'ord_7', 'ord_8', 'ord_9', 'ord_10', 'ord_11', 'ord_12',
           'ord_13', 'ord_14', 'ord_15', 'ord_16',
           'ord_17', 'ord_18', 'ord_19']

cat_col = ['cat_0', 'cat_1', 'cat_2', 'cat_3',
       'cat_4', 'cat_5', 'cat_6', 'cat_7']

num_pipeline = Pipeline(steps=[
    (
        'num_imputer',
        KNNImputer(missing_values=np.nan, n_neighbors=3)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipeline = Pipeline(steps=[
    # (
    #     'ord_imputer',
    #     SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=-1)        
    # )
    (
        'OE',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value = -1)  # be careful about the fill_value as the whole columns are stings and not numbers ...
    )
])

cat_pipeline = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OHE',
        OneHotEncoder(handle_unknown='ignore')
    )
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('numeric', num_pipeline, num_col)
    ,
    ('ordinal', ord_pipeline, ord_col)
    ,
    ('categorical', cat_pipeline, cat_col)
    ])


full_pipeline = Pipeline(steps=[
    ('preprocessing',preprocessing_pipeline),
    ('regressor', RidgeCV(alphas=[0.1, 1, 5, 10, 15, 20], cv=4))
])


full_pipeline.fit(x_train, y_train)

# get the test data --> useless they're only 100 rows >:(
test_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/test_dataset.csv'
test_df = pd.read_csv(test_path)
y_pred = full_pipeline.predict(test_df)

# turn your predictions into a csv file:
pred_df = pd.DataFrame({'ID': range(len(y_pred)), 'target': y_pred})
pred_csv = pred_df.to_csv('predictions.csv', index=False)

print(f'SANITY CHECK: predictions has shape {y_pred.shape} and the true target have shape {test_df.shape}')


# I have the solutions, compare to the predictions
solutions_df = pd.read_csv('submission_solutions.csv')
y_true = solutions_df['value'].values

print(mean_squared_error(y_true, y_pred))
print(mean_absolute_error(y_true, y_pred))
print(r2_score(y_true, y_pred))

---

# Next time try this
1.	Add the same row filtering as the professor (at least test it):  
- drop rows with too many NaNs (but do this only on training, not on test).

2.	Try a KNNRegressor pipeline too (since the professor is using it) so you have a comparable baseline.  


---

In [213]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

In [ ]:
train_path = "/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Code/Lab_5/Lab 05 - Regression Exam Simulation/train_dataset.csv"

df_train = pd.read_csv(train_path)
df_train.shape

- Inspect data in general:
    - I immediately see Nan values
    - Different kind of columns / type of data
    - 6000 rows and 59 columns

Inspect Nans

In [145]:
df = df_train.copy()
target = df['target']
df.drop(columns='target', inplace=True)

In [ ]:
# columns

na_count = df.isna().sum()
na_perc = df.isna().mean()

for col_idx, na_c, na_p in zip(na_count.index, na_count, na_perc):
    print(f"{col_idx} : {na_c} --> {na_p*100}%")

In [ ]:
df.isna().mean().sort_values(ascending=False)*100

In [ ]:
# rows inspection
df.isna().mean(axis=1).sort_values(ascending=False).head(100) * 100


- about ≈18-20% Nan in all columns (except target)
- about ≈32-40% Nan in top 100 worst rows

- before doin anything check if there are hidden Nan (empty text, invalid categories, values).
    - to do this inspection divide the columns based on their data type

In [ ]:
df.columns

In [ ]:
num_col = df.loc[:, 'cont_0':'cont_29']

ord_col = df.loc[:, 'ord_0':'ord_19']

cat_col = df.loc[:, 'cat_0':'cat_7']

# numerical columns with all unique values
# nothing weird
for col in num_col:
    print(df[col].value_counts().head(20))
    print(df[col].value_counts().tail(20))

# ord 8,9 cols have 2 values, check better with a histplot later
for col in ord_col:
    print(df[col].value_counts().head(6))
    print(df[col].value_counts().tail(6))
    print()

# cat 4,5,6 col has only one value, check better with a histplot later
for col in cat_col:
    print(df[col].value_counts().head(6))
    print(df[col].value_counts().tail(6))
    print()

- I didn't find any dirty value / hidden nan
- numerical columns seems ok
- cat 4,5,6 col has only one value --> check with histplot, potential drop
- ord 8,9 cols have 2 values --> same

- plot data frequency (histogram)

In [ ]:
plt.figure(figsize=(18,18))
for plt_idx, n_c in enumerate(num_col):
    plt.subplot(10,3,plt_idx+1)
    sns.histplot(x=df[n_c], bins='auto', kde=True)
    plt.title(f'frequency of {n_c}')

plt.tight_layout()
plt.show()
    

- numerical columns needs to be standardized
- cannot find evident anomaly points / outsider

In [ ]:
plt.figure(figsize=(18,18))
for plt_idx, n_c in enumerate(ord_col):
    plt.subplot(10,3,plt_idx+1)
    sns.histplot(x=df[n_c], bins='auto', kde=False)
    plt.title(f'frequency of {n_c}')

plt.tight_layout()
plt.show()

- seems ok
- except ord_1, ord_8 and ord_9, they have only 2 values but they're balanced ...

In [ ]:
plt.figure(figsize=(12,12))
for plt_idx, n_c in enumerate(cat_col):
    plt.subplot(4,2,plt_idx+1)
    sns.histplot(x=df[n_c], bins='auto', kde=False)
    plt.xticks(rotation=45)
    plt.title(f'frequency of {n_c}')

plt.tight_layout()
plt.show()

- drop cat_4, cat_5, cat_6

In [147]:
df.drop(columns=['cat_4', 'cat_5', 'cat_6'], inplace=True)

### Manage Nans
- we said we have ≈18-20% nan per column, not worth dropping
- ≈32-40% Nan in top 100 worst rows --> drop top K rows and impute the rest --> REMEMBER THAT YOU DROPPED ROWS WHEN PREDICTING, DON'T DO IT FOR THE TEST SET TOO, OR THE TARGET WON'T HAVE SAME POSITION WITH THE PROFESSOR'S PREDICTIONS! **REMEMBER TO DROP THOSE ROWS INDEX FROM TARGETS TOO**

> - SPLIT DATA INTO TRAINING AND VALIDATION SET

> Let's manage Nans based on what we had in mind for preprocessing:  
- numerical --> scaling standard scaler --> do a smart imputing, like KNNimputer
- ordinal --> ordinal encoder (remember to pass categories) --> deals Nan indipendently replacing them with -1
- categorical --> one hot encoding --> replace with 'unknown' so the dimensionality doesn't explode, but the model can still learn --> add missing value column flag (do it directly inside the OHE)

- tie everything up with a columntransformer
- use any regressor:
    - KNN regressor (distances might be a bit fucked up because of dimensionality)
    - logistic regressor
    - Ridge
    - Lasso (we don't have so many dimensions though to justify using it)
    - simpleregression
- tune it with a GridsearchCV

- drop worst rows fixing a reasonable threshold

In [ ]:
threshold = 35

# (df.isna().mean(axis=1)*100 > threshold).value_counts() check of how many rows I'm dropping

worst_rows = df.loc[df.isna().mean(axis=1)*100 > threshold, :]
df.drop(index=worst_rows.index, inplace=True)
target.drop(index=worst_rows.index, inplace=True)


print(df.shape, target.shape)

- build the pipelines

In [ ]:
for col in ord_col:
    print(df[col].value_counts())


- the ordinal encoder is not able to figure out the correct order of the ordinal columns
- compute it on your own and pass it

In [ ]:
unique_categories = []
for col in ord_col:
    cats = df[col].dropna().unique().tolist()
    unique_categories.append(cats)  # one list per column

for col, unique_cols in zip(ord_col, unique_categories):
    print(col, unique_cols)


- each ordinal column has a different number of unique values and the order is not universal like XL > S --> compute for each ord column the correct order
- pass to the ordinal encoder a list of lists, where inside the list each list is referred to each column ; each column has its own list where the order is specified.

In [ ]:
categories_per_col = []
for col in ord_col.columns:
    cats = df[col].dropna().unique().tolist()
    cats.sort(key=lambda s: int(s.split('_')[-1]))      # s being each value inside the list --> uses last part after final "_" which is the number
    categories_per_col.append(cats)

print(len(categories_per_col))  # must be 20, one per ordinal column which are 20

- the idea is fine, but that shitty ass OE doesn't work at all

In [206]:
num_col_names = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']

ord_col_names = ['ord_0', 'ord_1',
       'ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6', 'ord_7', 'ord_8', 'ord_9',
       'ord_10', 'ord_11', 'ord_12', 'ord_13', 'ord_14', 'ord_15', 'ord_16',
       'ord_17', 'ord_18', 'ord_19']

cat_col_names = ['cat_0', 'cat_1', 'cat_2', 'cat_3', 'cat_7']

In [ ]:
# numerical pipeline

num_pipe = Pipeline(
    steps=[
        (
            'KNN',
            KNNImputer(weights='uniform', add_indicator=True)         # tune n_neighbors, weights{‘uniform’, ‘distance’} + missing value column
        )
        ,
        (
            'scaler',
            StandardScaler()
        )
    ]
)


ord_pipe = Pipeline(
    steps=[
        (
            'imputer_ord',
            SimpleImputer(strategy="constant", fill_value="unknown", add_indicator=True)    # I added it so I can build the missing value flag column here too
        )
        ,
        (
            'OE',           # categories=categories_per_col doesn't work because OE is trash
            OrdinalEncoder(categories='auto', handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)  # also manages nan and replaces them with -1
        )
    ]
)


cat_pipe = Pipeline(
    steps=[
        (
            'imputer_cat',
            SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='unknown', add_indicator=True)     # manages nan replacing them with 'unknown' + missing value column
        )
        ,
        (
            'OHE',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False)     # sparse_output=False just to not mix data types, you could have kept it
        )
    ]
)

preprocessing_pipeline = ColumnTransformer(transformers=[
    ('num', num_pipe, num_col_names),
    ('ord', ord_pipe, ord_col_names),
    ('cat', cat_pipe, cat_col_names)
    ], remainder='passthrough'      # in this way the not specified columns still get used by the model
)

model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing_pipeline),
    ('regression', KNeighborsRegressor(n_jobs=-1))
])

param_grid={
    'preprocessing__num__KNN__n_neighbors': [5,15,30],
    # 'preprocessing__num__KNN__weights': ['uniform', 'distance']

    'regression__n_neighbors' : [5,10],
    'regression__weights' : ['uniform', 'distance']
}


x_train, x_val, y_train, y_val = train_test_split(df, target, test_size=0.2, train_size=0.8, random_state=42, shuffle=True)

grid = GridSearchCV(estimator=model_pipeline, param_grid=param_grid, scoring='r2', n_jobs=-1, cv=4, verbose=2)
grid.fit(x_train, y_train)
print(grid.best_params_)

best_model = grid.best_estimator_
y_pred = best_model.predict(x_val)
print(r2_score(y_val, y_pred))


---

# Mistakes + fixes
> you decide the rules using TRAIN only, then apply the same rules to VAL  
What you should have done (the correct mental model)  
- TRAIN = kitchen (you cook + choose the recipe)
- VAL = judge (you serve the dish; you don’t change the recipe based on the judge)

So:  
- fit / decide on TRAIN
- transform / apply on VAL (no fitting, no deciding)

__

#### Mistake 1 — Dropping rows before the split
What you did
- You computed “worst rows” (NaN% per row) on the full dataset
- You dropped those rows before train_test_split

Why it’s wrong
- Your TRAIN set got “cleaned” using information coming also from what later became VAL.
- That changes the distribution of TRAIN in a way that depends on VAL → leakage.

Correct way
1.	split first  
2.	compute the “drop rows” rule on TRAIN only  
3.	drop only from TRAIN (and y_train)  
4.	do not drop rows from VAL (otherwise your y_val alignment / evaluation becomes messy)  

__

#### Mistake 2 — Building ordinal category lists before the split
What you did
- You looked at all data to collect the unique ordinal tokens and their ordering.

Why it’s wrong
- You used VAL to decide which categories exist and (sometimes) their order.
- That makes VAL transformation “easier” (fewer unknown categories), inflating score.

Correct way
- Build categories_per_col using X_train only
- Then transform X_val using that fixed list:
- categories not seen in train become unknown_value (e.g., -1)

__

#### Mistake 3 — You broke the ordinal pipeline by adding extra columns before OrdinalEncoder
What you did: you used an imputer with add_indicator=True in the ordinal branch.  
Why it sucks: indicator adds new columns → OrdinalEncoder expects 20 columns but gets “20 + extra flags” → shape mismatch / errors.  
Fix: in ordinal branch:  
- do not add indicator columns
- just impute missing to a placeholder (like "missing") and encode

__

##### Mistake 4 — KNNRegressor + OneHot = distance becomes dumb

What you did: you used KNNRegressor after OneHot + other stuff.  
Why it sucks: in high-dim one-hot space, distances become noisy → model can act random-ish.  
Fix (easy): use a safer baseline regressor (Ridge / Linear) for “mixed columns + OHE”.  
Fix (if you insist on KNN): reduce dimensions first or avoid huge OHE explosion (not always possible in exams).  

__

Mistake 5 — remainder="passthrough" can sneak junk into the model (should be ok)  

What you did: you passthrough any column you forgot.  
Why it sucks: if you missed a column (or an ID-like column), it goes raw into the model and ruins training.  
Fix: prefer remainder="drop" unless you are 100% sure you listed every column.  